In [1]:
import pandas as pd
from azureml.core import Dataset

In [2]:
from azureml.core import Environment, Workspace
from azureml.core.conda_dependencies import CondaDependencies

ws = Workspace.from_config('.azureml/config')

If you run your code in unattended mode, i.e., where you can't give a user input, then we recommend to use ServicePrincipalAuthentication or MsiAuthentication.
Please refer to aka.ms/aml-notebook-auth for different authentication mechanisms in azureml-sdk.


In [83]:
from azureml.core import  ComputeTarget
from azureml.core.compute import AmlCompute

compute_name = 'AML-CC-01'
provisioning_config = AmlCompute.provisioning_configuration(vm_size='Standard_E16s_v3',
                                                            max_nodes=2,
                                                            )


compute_target = ComputeTarget.create(workspace=ws,
                                      name = compute_name,
                                      provisioning_configuration=provisioning_config)

In [61]:
dataset = Dataset.get_by_name(workspace=ws, name='Telecom_churn_dataset')

In [65]:
#df = dataset.to_pandas_dataframe()

df['Churn'].value_counts()

Churn
False    5163
True     1869
Name: count, dtype: int64

In [4]:
import joblib

# Use the absolute path to the joblib file
per_df = joblib.load(filename='E:/My Projects/Telecom customer churn-Azure ML SDK/jobs/joblib/automl_per_df.pkl')




In [53]:
dff=per_df.drop(['Run ID','Recall',	'Precision'],axis=1)
dff.sort_values(by='Normalized Recall',ascending=False)

,Algorithm,Accuracy,AUC,Normalized Recall
1,VotingEnsemble,0.769826,0.848070,0.531833
0,StackEnsemble,0.758452,0.847804,0.528032
17,RandomForest,0.747709,0.844200,0.524269
12,RandomForest,0.746130,0.842271,0.518614
23,RandomForest,0.742338,0.838712,0.514194
11,ExtremeRandomTrees,0.745656,0.841018,0.513560
20,SGD,0.732070,0.841975,0.513095
19,RandomForest,0.744708,0.834171,0.512053
22,RandomForest,0.731438,0.833166,0.510607
26,ExtremeRandomTrees,0.740758,0.837568,0.510476


In [38]:
# Assuming dff is your DataFrame containing the model performance metrics
filtered_models = dff[(dff['AUC'] > 0.80) & (dff['Normalized Recall'] > 0.20) & (dff['Accuracy'] > 70)]

# Display the filtered models
print(filtered_models)


Empty DataFrame
Columns: [Algorithm, Accuracy, AUC, Normalized Recall]
Index: []


In [24]:
from azureml.core import Workspace, Experiment, Model
from azureml.train.hyperdrive import HyperDriveRun


# Initialize workspace
ws = Workspace.from_config('.azureml/config')

# Define the HyperDrive parent run ID
parent_run_id = 'HD_cc60266b-50a1-4534-a242-10edc5ef6215'

experiment = Experiment(ws,name='Hyperdrive_voting_classifier_Telecome_01')


parent_run = HyperDriveRun(experiment=experiment,
                            run_id=parent_run_id)

best_run = parent_run.get_best_run_by_primary_metric()
best_run_id = best_run.id


model = best_run.register_model(model_name='Telecom_voting_classifier',
                                model_path='outputs/voting_classifier_model.pkl',
                                tags = {'Source' : 'Hyperdriver best run','Algorithm' : 'Voting_classifier'},
                                properties = {
                                    'accuracy': best_run_metrics.get('accuracy'),
                                    'Recall': best_run_metrics.get('recall'),
                                    'Precision': best_run_metrics.get('precision'),
                                    'f1_score': best_run_metrics.get('f1'),
                                    'AUC_Score': best_run_metrics.get('auc'),
                                    'Best Hyperparameters': str(hyperparameters)},
                                description = 'Voting classifier with Random forest and gradient boosting algoritm are used')

In [23]:


best_run_id

'HD_cc60266b-50a1-4534-a242-10edc5ef6215_152'